In [3]:
from langgraph.graph import StateGraph, START, END
from content_agent.graph.state import State

In [5]:
def fake_searcher(state: State) -> dict:
    print(f"🔍 Searcher received topic: {state['topic']}")
    return {
        "search_results": [
            {"url": "https://example.com/a", "title": "Article A", "snippet": "..."},
        ],
        "status": "researching",
    }

In [8]:
def fake_critic(state: State) -> dict:
    count = state.get("revision_count", 0) + 1
    print(f"🧐 Critic pass #{count}")
    return {
        "critic_notes": [f"note from pass {count}"],   # appended, thanks to the reducer
        "revision_count": count,                        # overwritten
    }

In [9]:
def should_loop(state: State) -> str:
    """Decides where to go after the critic."""
    return "critic" if state["revision_count"] < 3 else END

In [11]:
builder = StateGraph(State)
builder.add_node("searcher", fake_searcher)
builder.add_node("critic", fake_critic)
builder.add_edge(START, "searcher")
builder.add_edge("searcher", "critic")
builder.add_conditional_edges("critic", should_loop)

graph = builder.compile()

result = graph.invoke({"topic": "Rust in the Linux kernel", "topic_source": "user", "guidance": None})
result

🔍 Searcher received topic: Rust in the Linux kernel
🧐 Critic pass #1
🧐 Critic pass #2
🧐 Critic pass #3


{'topic': 'Rust in the Linux kernel',
 'topic_source': 'user',
 'guidance': None,
 'search_results': [{'url': 'https://example.com/a',
   'title': 'Article A',
   'snippet': '...'}],
 'critic_notes': ['note from pass 1', 'note from pass 2', 'note from pass 3'],
 'revision_count': 3,
 'status': 'researching'}

In [12]:
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	searcher(searcher)
	critic(critic)
	__end__([<p>__end__</p>]):::last
	__start__ --> searcher;
	searcher --> critic;
	critic --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

